## Pipeline ML — Predicción de Morosidad Bancaria

Este notebook implementa un Pipeline completo con `sklearn.pipeline.Pipeline` para predecir la morosidad de clientes bancarios, integrando en un único flujo reproducible la limpieza, preprocesamiento, entrenamiento y predicción del modelo.

## Importación de librerías y carga de datos

Se importan las librerías necesarias y se carga el dataset original.

In [2]:
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Cargar datos
df = pd.read_csv("data/data.csv")
df.head()

,mora,atraso,vivienda,edad,dias_lab,exp_sf,nivel_ahorro,ingreso,linea_sf,deuda_sf,score,zona,clasif_sbs,nivel_educ
0,0,235,FAMILIAR,30,3748,93.0,5,3500.0,NaN,0.00,214,Lima,4,UNIVERSITARIA
1,0,18,FAMILIAR,32,4598,9.0,12,900.0,1824.67,1933.75,175,La Libertad,1,TECNICA
2,0,0,FAMILIAR,26,5148,8.0,2,2400.0,2797.38,188.29,187,Lima,0,UNIVERSITARIA
3,0,0,FAMILIAR,36,5179,20.0,12,2700.0,NaN,0.00,187,Ancash,0,TECNICA
4,0,0,FAMILIAR,46,3960,NaN,1,3100.0,2000.00,11010.65,189,Lima,0,TECNICA


## Preparación previa y Transformer personalizado

Dado que `drop_duplicates` modifica el número de filas del dataset, esta operación se realiza **fuera del pipeline** a través de la función `preparar_datos()`, que retorna `X` e `y` listos para el split.

El transformer personalizado `MorosidadCleaner` se encarga de dos tareas dentro del pipeline:

- **Imputación KNN:** rellena los valores nulos de `exp_sf`, `linea_sf` y `deuda_sf` ajustándose exclusivamente sobre `X_train`, evitando *data leakage*.
- **Codificación ordinal:** transforma `nivel_educ` a valores numéricos respetando su jerarquía lógica.

In [3]:
# Mapeo de educación y función de preparación de datos
educacion_mapping = {
    'SIN EDUCACION': 0,
    'PRIMARIA'     : 1,
    'SECUNDARIA'   : 2,
    'TECNICA'      : 3,
    'UNIVERSITARIA': 4,
    'POSTGRADO'    : 5
}

class MorosidadCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, educacion_mapping):
        self.educacion_mapping = educacion_mapping
        self.imputer = KNNImputer(n_neighbors=5)
    
    def fit(self, X, y=None):
        self.imputer.fit(X[['exp_sf', 'linea_sf', 'deuda_sf']])
        return self
    
    def transform(self, X):
        df = X.copy()
        df[['exp_sf', 'linea_sf', 'deuda_sf']] = self.imputer.transform(
            df[['exp_sf', 'linea_sf', 'deuda_sf']]
        )
        df['nivel_educ'] = df['nivel_educ'].map(self.educacion_mapping).fillna(0)
        return df

def preparar_datos(df):
    df_clean = df.drop_duplicates(ignore_index=True).copy()
    X = df_clean.drop(columns=['mora'])
    y = df_clean['mora']
    return X, y

X, y = preparar_datos(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

## Definición de columnas y preprocesador

Se definen las columnas según su naturaleza y se construye el `ColumnTransformer` que aplica transformaciones distintas a cada grupo:

- **`StandardScaler`** sobre las variables numéricas, incluyendo `nivel_educ` y `clasif_sbs` que, al ser variables ordinales con jerarquía lógica, se tratan como numéricas.
- **`OneHotEncoder`** con `drop='first'` únicamente sobre `vivienda` y `zona`, variables nominales sin orden entre sus categorías, evitando la **multicolinealidad** (*dummy variable trap*).

In [4]:
# Definir columnas - clasif_sbs movida a numéricas (es ordinal)
cols_num = ['atraso', 'edad', 'dias_lab', 'exp_sf', 'ingreso',
            'linea_sf', 'deuda_sf', 'score', 'nivel_ahorro',
            'nivel_educ', 'clasif_sbs'] 

cols_ohe = ['vivienda', 'zona']

# Pipeline numérico
numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Pipeline categórico
categorical_pipeline = Pipeline([
    ('ohe', OneHotEncoder(drop='first', sparse_output=False))
])

# ColumnTransformer
feature_preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, cols_num),
    ('cat', categorical_pipeline, cols_ohe)
], remainder='drop')

## Pipeline completo

Se ensambla el pipeline completo con tres etapas en orden:

1. `cleaning` — limpieza e imputación con `MorosidadCleaner`
2. `preprocessing` — escalado y codificación con `ColumnTransformer`
3. `classifier` — modelo **Random Forest** con `class_weight='balanced'`

In [5]:
# Pipeline completo con orden correcto
complete_pipeline = Pipeline([
    ('cleaning', MorosidadCleaner(educacion_mapping)),
    ('preprocessing', feature_preprocessor),
    ('classifier', RandomForestClassifier(
                        n_estimators=300,
                        class_weight='balanced',
                        random_state=42))
])

print("\nPipeline completo:")
print(complete_pipeline)


Pipeline completo:
Pipeline(steps=[('cleaning',
                 MorosidadCleaner(educacion_mapping={'POSTGRADO': 5,
                                                     'PRIMARIA': 1,
                                                     'SECUNDARIA': 2,
                                                     'SIN EDUCACION': 0,
                                                     'TECNICA': 3,
                                                     'UNIVERSITARIA': 4})),
                ('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['atraso', 'edad', 'dias_lab',
                                                   'exp_sf', 'ingreso',
                                                   'linea_sf', 'deuda_sf',
                                            

## Entrenamiento

Se entrena el pipeline completo sobre `X_train`. Internamente, cada etapa ejecuta su método `fit` en secuencia, garantizando que ninguna transformación utilice información del conjunto de prueba.

In [6]:
# Entrenamiento del modelo
print("\nEntrenando modelo...")
complete_pipeline.fit(X_train, y_train)
print("¡Entrenamiento completado!")


Entrenando modelo...
¡Entrenamiento completado!


## Evaluación del modelo

Se evalúa el rendimiento del pipeline sobre `X_test`, generando tanto las etiquetas predichas (`y_pred`) como las probabilidades (`y_proba`) para el cálculo del ROC-AUC.

In [7]:
# Evaluación
print("\nEvaluando modelo...")
y_pred  = complete_pipeline.predict(X_test)
y_proba = complete_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba))


Evaluando modelo...
Accuracy : 0.77119341563786
Precision: 0.7804410354745925
Recall   : 0.9432213209733488
F1-score : 0.8541448058761805
ROC-AUC  : 0.8240035420836406


## Guardar pipeline

Se serializa el pipeline completo con `joblib`, guardando en un único archivo todas las etapas: transformer, preprocesador y modelo entrenado. Esto permite reutilizarlo en producción sin necesidad de reentrenar.

In [8]:
# Guardar pipeline
print("\nGuardando pipeline...")
joblib.dump(complete_pipeline, 'pipeline_morosidad.pkl')
print("Archivo guardado: pipeline_morosidad.pkl")


Guardando pipeline...
Archivo guardado: pipeline_morosidad.pkl


## Predicción sobre nuevos datos

Se carga el pipeline guardado y se realiza una predicción sobre un cliente nuevo enviado en formato **JSON**. El pipeline aplica automáticamente todas las transformaciones aprendidas durante el entrenamiento antes de generar la predicción.

In [9]:
# Predicción sobre nuevos datos (JSON)
import json

pipeline_loaded = joblib.load('pipeline_morosidad.pkl')

nuevo_cliente_json = '''
{
    "atraso": 5,
    "edad": 35,
    "dias_lab": 720,
    "exp_sf": 24,
    "ingreso": 3500.0,
    "linea_sf": 10000.0,
    "deuda_sf": 2000.0,
    "score": 650,
    "nivel_ahorro": 6,
    "nivel_educ": "UNIVERSITARIA",
    "vivienda": "PROPIA",
    "zona": "Lima",
    "clasif_sbs": 0
}
'''

cliente_df = pd.DataFrame([json.loads(nuevo_cliente_json)])

prediccion   = pipeline_loaded.predict(cliente_df)
probabilidad = pipeline_loaded.predict_proba(cliente_df)[0][1]

print("Mora predicha  :", "Cliente Moroso" if prediccion[0] == 1 else "Paga al día")
print(f"Probabilidad de mora: {probabilidad:.2%}")

Mora predicha  : Cliente Moroso
Probabilidad de mora: 79.33%
